# 03 - Feature Engineering e Dataset Analitico

Objetivo deste notebook: criar uma tabela analitica consolidada para responder perguntas de negocio e servir como base para dashboard.

Vamos juntar dados de pedidos, clientes, itens, produtos, categorias, pagamentos e avaliacoes.

Ao final, salvaremos um arquivo tratado em `data/processed/orders_analytics.csv`.

## 1. Importar bibliotecas

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 140)

## 2. Definir caminhos e carregar dados

In [ ]:
PROJECT_ROOT = Path("..").resolve()
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

orders = pd.read_csv(RAW_DATA_DIR / "olist_orders_dataset.csv")
items = pd.read_csv(RAW_DATA_DIR / "olist_order_items_dataset.csv")
payments = pd.read_csv(RAW_DATA_DIR / "olist_order_payments_dataset.csv")
reviews = pd.read_csv(RAW_DATA_DIR / "olist_order_reviews_dataset.csv")
products = pd.read_csv(RAW_DATA_DIR / "olist_products_dataset.csv")
customers = pd.read_csv(RAW_DATA_DIR / "olist_customers_dataset.csv")
category_translation = pd.read_csv(RAW_DATA_DIR / "product_category_name_translation.csv")

orders.shape, items.shape, payments.shape, reviews.shape, products.shape, customers.shape

## 3. Converter datas

As colunas de data precisam estar em formato `datetime` para calcular prazos e atrasos.

In [ ]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

for column in date_columns:
    orders[column] = pd.to_datetime(orders[column])

orders[date_columns].dtypes

## 4. Criar resumo de itens por pedido

A tabela `items` esta no nivel item. Para criar uma tabela no nivel pedido, precisamos agregar por `order_id`.

Aqui calculamos receita de produtos, frete, quantidade de itens e quantidade de vendedores por pedido.

In [ ]:
order_items_summary = (
    items
    .groupby("order_id")
    .agg(
        product_revenue=("price", "sum"),
        freight_value=("freight_value", "sum"),
        item_count=("order_item_id", "count"),
        seller_count=("seller_id", "nunique"),
    )
    .reset_index()
)

order_items_summary.head()

## 5. Definir categoria principal do pedido

Um pedido pode ter mais de um item. Para simplificar a analise inicial, vamos usar como categoria principal a categoria do item de maior preco dentro do pedido.

In [ ]:
items_with_categories = (
    items
    .merge(products[["product_id", "product_category_name"]], on="product_id", how="left")
    .merge(category_translation, on="product_category_name", how="left")
)

items_with_categories["category"] = items_with_categories["product_category_name_english"].fillna("unknown")

main_category_by_order = (
    items_with_categories
    .sort_values(["order_id", "price"], ascending=[True, False])
    .drop_duplicates("order_id")
    [["order_id", "category"]]
    .rename(columns={"category": "main_category"})
)

main_category_by_order.head()

## 6. Criar resumo de pagamentos por pedido

A tabela de pagamentos tambem pode ter mais de uma linha por pedido. Vamos agregar valor pago, parcelas maximas e tipo principal de pagamento.

In [ ]:
payment_type_by_order = (
    payments
    .sort_values(["order_id", "payment_value"], ascending=[True, False])
    .drop_duplicates("order_id")
    [["order_id", "payment_type"]]
    .rename(columns={"payment_type": "main_payment_type"})
)

payments_summary = (
    payments
    .groupby("order_id")
    .agg(
        payment_value=("payment_value", "sum"),
        max_installments=("payment_installments", "max"),
        payment_records=("payment_sequential", "count"),
    )
    .reset_index()
    .merge(payment_type_by_order, on="order_id", how="left")
)

payments_summary.head()

## 7. Criar resumo de avaliacoes por pedido

Vamos manter a nota media e a quantidade de avaliacoes por pedido.

In [ ]:
reviews_summary = (
    reviews
    .groupby("order_id")
    .agg(
        review_score=("review_score", "mean"),
        review_count=("review_id", "count"),
    )
    .reset_index()
)

reviews_summary.head()

## 8. Montar tabela analitica

Agora juntamos tudo no nivel pedido. Cada linha da tabela final deve representar um pedido.

In [ ]:
orders_analytics = (
    orders
    .merge(customers, on="customer_id", how="left")
    .merge(order_items_summary, on="order_id", how="left")
    .merge(main_category_by_order, on="order_id", how="left")
    .merge(payments_summary, on="order_id", how="left")
    .merge(reviews_summary, on="order_id", how="left")
)

orders_analytics.shape

## 9. Criar features de tempo, entrega e atraso

In [ ]:
orders_analytics["order_year"] = orders_analytics["order_purchase_timestamp"].dt.year
orders_analytics["order_month"] = orders_analytics["order_purchase_timestamp"].dt.month
orders_analytics["order_year_month"] = orders_analytics["order_purchase_timestamp"].dt.to_period("M").astype(str)

orders_analytics["approval_days"] = (
    orders_analytics["order_approved_at"] - orders_analytics["order_purchase_timestamp"]
).dt.total_seconds() / 86400

orders_analytics["delivery_days"] = (
    orders_analytics["order_delivered_customer_date"] - orders_analytics["order_purchase_timestamp"]
).dt.total_seconds() / 86400

orders_analytics["estimated_delivery_days"] = (
    orders_analytics["order_estimated_delivery_date"] - orders_analytics["order_purchase_timestamp"]
).dt.total_seconds() / 86400

orders_analytics["delay_days"] = (
    orders_analytics["order_delivered_customer_date"].dt.normalize()
    - orders_analytics["order_estimated_delivery_date"].dt.normalize()
).dt.days

orders_analytics["is_late"] = np.where(
    orders_analytics["order_delivered_customer_date"].notna(),
    orders_analytics["delay_days"] > 0,
    np.nan,
)

orders_analytics[["order_id", "order_status", "order_year_month", "delivery_days", "delay_days", "is_late"]].head()

## 10. Validacoes de qualidade

Antes de salvar, precisamos garantir que nao duplicamos pedidos ao fazer os joins.

In [ ]:
quality_checks = pd.DataFrame({
    "metric": [
        "rows_orders_original",
        "rows_orders_analytics",
        "unique_orders_original",
        "unique_orders_analytics",
        "duplicated_order_ids",
    ],
    "value": [
        len(orders),
        len(orders_analytics),
        orders["order_id"].nunique(),
        orders_analytics["order_id"].nunique(),
        orders_analytics["order_id"].duplicated().sum(),
    ]
})

quality_checks

## 11. Selecionar colunas finais

Vamos manter uma tabela enxuta, mas com informacoes suficientes para analise e dashboard.

In [ ]:
final_columns = [
    "order_id",
    "customer_id",
    "customer_unique_id",
    "customer_city",
    "customer_state",
    "order_status",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "order_year",
    "order_month",
    "order_year_month",
    "product_revenue",
    "freight_value",
    "payment_value",
    "item_count",
    "seller_count",
    "main_category",
    "main_payment_type",
    "max_installments",
    "payment_records",
    "review_score",
    "review_count",
    "approval_days",
    "delivery_days",
    "estimated_delivery_days",
    "delay_days",
    "is_late",
]

orders_analytics_final = orders_analytics[final_columns].copy()
orders_analytics_final.head()

## 12. Salvar dataset processado

In [ ]:
output_path = PROCESSED_DATA_DIR / "orders_analytics.csv"
orders_analytics_final.to_csv(output_path, index=False)

output_path

## 13. Resumo final da tabela criada

In [ ]:
processed_summary = pd.DataFrame({
    "rows": [orders_analytics_final.shape[0]],
    "columns": [orders_analytics_final.shape[1]],
    "unique_orders": [orders_analytics_final["order_id"].nunique()],
    "file": [str(output_path)],
})

processed_summary

## 14. O que fizemos

Neste notebook, criamos uma tabela no nivel pedido com informacoes comerciais, logisticas, geograficas, de pagamento e avaliacao.

Essa tabela sera usada nas proximas etapas para:

1. responder perguntas de negocio com mais velocidade;
2. criar consultas SQL;
3. montar um dashboard no Power BI;
4. documentar insights no README do projeto.